In [ ]:
# ============================================================
# Step 1: Run Moondream and save generated descriptions
# Kaggle-compatible: rebuilds image paths, writes to /kaggle/working
# ============================================================

import os
import time
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import moondream as md
from kaggle_secrets import UserSecretsClient

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INPUT_CSV           = '/kaggle/input/datasets/salmankey/dataset-ent-images-with-descriptions-2/dataset_ent_images_with_descriptions (2).csv'
OUTPUT_WITH_GEN_CSV = '/kaggle/working/ent_with_generated_descriptions1.csv'

IMAGE_DIR = '/kaggle/input/datasets/salmankey/dataset-en/dataset_ent'

MOONDREAM_MODEL = "moondream3.1-9B-A2B"

# Prompt used to generate descriptions. Keep identical across all images
GENERATION_PROMPT = """
You are a clinical image describer. Describe only what is visibly present in the image.
Identify the visible anatomical area when it can be reasonably recognized (external ear,
ear canal, eardrum, nose, nasal cavity, mouth, throat). Describe visible anatomical
structures, colours, shapes, textures, and relative positions. When appropriate, describe
yellowish or brownish material, reddish or pale areas, dark regions, apparent swelling,
fluid-like material, crusting, or other visible features. Describe what an area looks like
rather than claiming what it represents. If the image is blurry, overexposed, or obscured,
explicitly say so instead of guessing.

Output format: write ONE short paragraph of 3-5 sentences in simple, everyday English.
Use continuous prose. Do not diagnose, name a disease, infer a cause, recommend
treatment, or invent findings.
""".strip()

# ------------------------------------------------------------
# Secrets
# ------------------------------------------------------------

user_secrets = UserSecretsClient()

MOONDREAM_API_KEY = user_secrets.get_secret("Moondream_testing")
if not MOONDREAM_API_KEY:
    raise ValueError("Moondream_testing secret is empty or missing.")

# Optional: HF token — only needed if you download models from HuggingFace
# HF_TOKEN = user_secrets.get_secret("HF_TOKEN")  # uncomment if needed

# ------------------------------------------------------------
# Load CSV and rebuild image paths
# ------------------------------------------------------------

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows from {INPUT_CSV}")
print(f"Existing columns: {list(df.columns)}")
print(f"First path BEFORE fix: {df.iloc[0]['path']}")

# Overwrite 'path' to point to the Kaggle image directory
df['path'] = df['image_name'].apply(
    lambda name: os.path.join(IMAGE_DIR, str(name))
)
print(f"First path AFTER fix:  {df.iloc[0]['path']}")

# Verify all images exist before calling the API
df['_exists'] = df['path'].apply(os.path.isfile)
n_missing = int((~df['_exists']).sum())
if n_missing > 0:
    print(f"\n⚠️  {n_missing} image(s) not found at expected path:")
    print(df.loc[~df['_exists'], ['image_name', 'path']].to_string(index=False))
    # Drop missing rows so the run does not crash mid-way
    df = df[df['_exists']].reset_index(drop=True)
    print(f"Continuing with {len(df)} existing images.")
df = df.drop(columns=['_exists'])

if len(df) == 0:
    raise ValueError("No valid images found. Check IMAGE_DIR.")

# ------------------------------------------------------------
# Run Moondream
# ------------------------------------------------------------

client = md.vl(api_key=MOONDREAM_API_KEY, model=MOONDREAM_MODEL)

generated = []
statuses  = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Moondream"):
    try:
        with Image.open(row['path']) as img:
            image = img.convert("RGB")
        result = client.query(image=image, question=GENERATION_PROMPT)
        ans = result.get('answer', '') if isinstance(result, dict) else str(result)
        ans = str(ans).strip().strip('"').strip()
        if ans.lower().startswith('answer:'):
            ans = ans.split(':', 1)[1].strip()
        generated.append(ans)
        statuses.append('success' if ans else 'failed')
    except Exception as e:
        print(f"\nError on {row['image_name']}: {e}")
        generated.append("")
        statuses.append('failed')
    time.sleep(1)

df['description_generated']        = generated
df['description_generated_status'] = statuses

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

df.to_csv(OUTPUT_WITH_GEN_CSV, index=False)
print(f"\nSaved: {OUTPUT_WITH_GEN_CSV}")
print(f"Successful generations: {(df['description_generated_status'] == 'success').sum()}/{len(df)}")

print("\nFirst row preview:")
print(f"  image_name:             {df.iloc[0]['image_name']}")
print(f"  description (GT):       {df.iloc[0]['description'][:80]}...")
print(f"  description_generated:  {df.iloc[0]['description_generated'][:80]}...")

In [ ]:
# ============================================================
# Step 2: Dual-Model Cosine Similarity — GT vs Generated
# BGE-large (general) vs MedEmbed (medical-specific)
# ============================================================

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

INPUT_CSV   = '/kaggle/working/ent_with_generated_descriptions1.csv'
OUTPUT_CSV  = '/kaggle/working/ent_cosine_eval_dual.csv'
SUMMARY_CSV = '/kaggle/working/ent_cosine_eval_dual_summary.csv'

GENERAL_MODEL   = 'BAAI/bge-large-en-v1.5'
MEDICAL_MODEL   = 'abhinand/MedEmbed-base-v0.1'
MATCH_THRESHOLD = 0.75

GT_COLUMN  = 'description'
GEN_COLUMN = 'description_generated'

# ------------------------------------------------------------
# Load and filter
# ------------------------------------------------------------

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows")

df = df[
    df[GEN_COLUMN].astype(str).str.strip().ne('') &
    df[GEN_COLUMN].notna() &
    df[GT_COLUMN].astype(str).str.strip().ne('') &
    df[GT_COLUMN].notna()
].copy().reset_index(drop=True)
print(f"Valid rows: {len(df)}")

df['gt_words']  = df[GT_COLUMN].apply(lambda s: len(str(s).split()))
df['gen_words'] = df[GEN_COLUMN].apply(lambda s: len(str(s).split()))

# ------------------------------------------------------------
# Score with a given model
# ------------------------------------------------------------

def score_model(model_name, tag):
    print(f"\nLoading {model_name} ...")
    model = SentenceTransformer(model_name)

    gt_emb = model.encode(
        df[GT_COLUMN].tolist(),
        convert_to_tensor=True, normalize_embeddings=True,
    )
    gen_emb = model.encode(
        df[GEN_COLUMN].tolist(),
        convert_to_tensor=True, normalize_embeddings=True,
    )
    sims = util.cos_sim(gt_emb, gen_emb).diagonal().cpu().numpy()

    df[f'sim_{tag}'] = sims
    print(f"  mean={sims.mean():.4f}  std={sims.std():.4f}  "
          f"min={sims.min():.4f}  max={sims.max():.4f}  "
          f"match@0.75={(sims >= MATCH_THRESHOLD).sum()}/{len(sims)}")
    return sims

sims_bge = score_model(GENERAL_MODEL, 'bge')
sims_med = score_model(MEDICAL_MODEL, 'medembed')

# ------------------------------------------------------------
# Per-image comparison
# ------------------------------------------------------------

df['diff_med_minus_bge'] = df['sim_medembed'] - df['sim_bge']
df['winner'] = df.apply(
    lambda r: 'MedEmbed' if r['sim_medembed'] > r['sim_bge']
    else ('BGE' if r['sim_bge'] > r['sim_medembed'] else 'Tie'),
    axis=1
)

print("\n===== PER-IMAGE COMPARISON =====")
preview = ['image_name', 'sim_bge', 'sim_medembed', 'diff_med_minus_bge', 'winner']
print(df[preview].sort_values('diff_med_minus_bge', ascending=False).to_string(index=False))

# ------------------------------------------------------------
# Which model is better?
# ------------------------------------------------------------

print("\n===== HEAD-TO-HEAD VERDICT =====")

# Criterion 1: higher mean similarity
bge_mean = df['sim_bge'].mean()
med_mean = df['sim_medembed'].mean()
print(f"Mean similarity:     BGE={bge_mean:.4f}  vs  MedEmbed={med_mean:.4f}")
print(f"  → Higher mean:      {'MedEmbed' if med_mean > bge_mean else 'BGE'}")

# Criterion 2: higher match rate
bge_match = (df['sim_bge'] >= MATCH_THRESHOLD).sum()
med_match = (df['sim_medembed'] >= MATCH_THRESHOLD).sum()
print(f"Match rate @{MATCH_THRESHOLD}:  BGE={bge_match}/{len(df)}  vs  MedEmbed={med_match}/{len(df)}")
print(f"  → Higher match rate: {'MedEmbed' if med_match > bge_match else ('BGE' if bge_match > med_match else 'Tie')}")

# Criterion 3: win count
n_med_wins = (df['winner'] == 'MedEmbed').sum()
n_bge_wins = (df['winner'] == 'BGE').sum()
n_ties     = (df['winner'] == 'Tie').sum()
print(f"Per-image wins:      MedEmbed={n_med_wins}  BGE={n_bge_wins}  Tie={n_ties}")
print(f"  → More wins:        {'MedEmbed' if n_med_wins > n_bge_wins else ('BGE' if n_bge_wins > n_med_wins else 'Tie')}")

# Criterion 4: separation (higher = descriptions better distinguished)
bge_sep = df['sim_bge'].std()
med_sep = df['sim_medembed'].std()
print(f"Std (spread):        BGE={bge_sep:.4f}  vs  MedEmbed={med_sep:.4f}")

# Overall verdict
med_score = sum([med_mean > bge_mean, med_match >= bge_match, n_med_wins >= n_bge_wins])
print(f"\nOverall verdict:     {'MedEmbed' if med_score >= 2 else 'BGE'} "
      f"(MedEmbed wins {med_score}/3 primary criteria)")

# ------------------------------------------------------------
# Save per-image results
# ------------------------------------------------------------

out_cols = [
    'image_name',
    'sim_bge', 'sim_medembed', 'diff_med_minus_bge', 'winner',
    'gt_words', 'gen_words',
    'description', 'description_generated',
]
df[out_cols].to_csv(OUTPUT_CSV, index=False)
print(f"\nPer-image results saved: {OUTPUT_CSV}")

# ------------------------------------------------------------
# Summary CSV
# ------------------------------------------------------------

summary = pd.DataFrame([
    {
        'model': GENERAL_MODEL,
        'type': 'general-purpose',
        'n': len(df),
        'mean': round(bge_mean, 4),
        'std': round(df['sim_bge'].std(), 4),
        'min': round(df['sim_bge'].min(), 4),
        'max': round(df['sim_bge'].max(), 4),
        'matches_at_0.75': bge_match,
        'match_rate': round(bge_match / len(df), 4),
        'per_image_wins': n_bge_wins,
    },
    {
        'model': MEDICAL_MODEL,
        'type': 'medical-specific',
        'n': len(df),
        'mean': round(med_mean, 4),
        'std': round(df['sim_medembed'].std(), 4),
        'min': round(df['sim_medembed'].min(), 4),
        'max': round(df['sim_medembed'].max(), 4),
        'matches_at_0.75': med_match,
        'match_rate': round(med_match / len(df), 4),
        'per_image_wins': n_med_wins,
    },
])
summary.to_csv(SUMMARY_CSV, index=False)
print(f"\nSummary saved: {SUMMARY_CSV}")
print("\n===== SUMMARY TABLE =====")
print(summary.to_string(index=False))

# ------------------------------------------------------------
# Threshold sensitivity for both
# ------------------------------------------------------------

print("\n===== THRESHOLD SENSITIVITY =====")
print(f"{'Threshold':>10}  {'BGE match':>12}  {'MedEmbed match':>15}")
for t in [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    b = int((df['sim_bge'] >= t).sum())
    m = int((df['sim_medembed'] >= t).sum())
    print(f"{t:>10.2f}  {b:>8}/{len(df):<3}  {m:>11}/{len(df):<3}")